Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import json, math, editdistance

Vocab + model classes

In [2]:
with open('../src/vocab.json') as f:
    vocab = json.load(f)
stoi = vocab['stoi']
itos = {int(k): v for k, v in vocab['itos'].items()}
PAD, BOS, EOS = 0, 1, 2
vocab_size = 39
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class CNNEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(), nn.MaxPool2d((2, 1)),
        )
    def forward(self, x):
        x = self.conv(x)
        x = x.squeeze(2)
        return x.permute(0, 2, 1)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.dk = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, q_in, kv_in, mask=None):
        B, Tq, _ = q_in.shape
        Tk = kv_in.shape[1]
        Q = self.q_proj(q_in).view(B, Tq, self.h, self.dk).transpose(1, 2)
        K = self.k_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)
        V = self.v_proj(kv_in).view(B, Tk, self.h, self.dk).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.dk)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = scores.softmax(dim=-1)
        out = attn @ V
        out = out.transpose(1, 2).contiguous().view(B, Tq, -1)
        return self.out_proj(out)

def causal_mask(T, device):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, ff_dim), nn.GELU(), nn.Linear(ff_dim, d_model))
    def forward(self, x):
        return self.net(x)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, ff_dim)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
    def forward(self, x, memory, self_mask):
        normed = self.norm1(x)
        x = x + self.self_attn(normed, normed, mask=self_mask)
        x = x + self.cross_attn(self.norm2(x), memory, mask=None)
        x = x + self.ffn(self.norm3(x))
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024, max_len=50):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, ff_dim) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
    def forward(self, tgt_in, memory):
        B, T = tgt_in.shape
        x = self.embed(tgt_in) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        mask = causal_mask(T, tgt_in.device)
        for layer in self.layers:
            x = layer(x, memory, mask)
        x = self.norm_out(x)
        return self.fc_out(x)

class OCRModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, n_layers=3, ff_dim=1024):
        super().__init__()
        self.encoder = CNNEncoder(d_model)
        self.decoder = Decoder(vocab_size, d_model, n_heads, n_layers, ff_dim)

    def forward(self, imgs, tgt_in):
        memory = self.encoder(imgs)
        return self.decoder(tgt_in, memory)

Load the trained weights

In [3]:
model = OCRModel(vocab_size=vocab_size).to(device)
model.load_state_dict(torch.load('../checkpoints/model_50k.pt', map_location=device))
model.eval()

OCRModel(
  (encoder): CNNEncoder(
    (conv): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (6): ReLU()
      (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (10): ReLU()
      (11): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 

Autoregression

In [4]:
@torch.no_grad()
def generate(model, img, max_len=25):
    model.eval()
    img = img.unsqueeze(0).to(device)          # add batch dim: [1, 1, 32, 128]
    memory = model.encoder(img)                 # [1, 32, 256]

    seq = [BOS]
    for _ in range(max_len):
        tgt = torch.tensor([seq], device=device)
        logits = model.decoder(tgt, memory)      # [1, len(seq), vocab_size]
        next_id = logits[0, -1].argmax().item()   # only look at LAST position
        if next_id == EOS:
            break
        seq.append(next_id)

    return ''.join(itos[i] for i in seq[1:])       # drop BOS

Testing on img 0

In [5]:
transform = T.Compose([
    T.Resize((32, 128)), T.Grayscale(), T.ToTensor(), T.Normalize([0.5], [0.5]),
])

img = transform(Image.open('../data/synth/train/00000.png').convert('RGB'))
pred = generate(model, img)
print(pred)

ot2be1yq


Metrics

In [6]:
def normalize(s):
    return ''.join(c for c in s.lower() if c.isalnum())

def word_accuracy(preds, labels):
    correct = sum(normalize(p) == normalize(l) for p, l in zip(preds, labels))
    return correct / len(labels)

def cer(preds, labels):
    total_dist = sum(editdistance.eval(normalize(p), normalize(l)) for p, l in zip(preds, labels))
    total_chars = sum(len(normalize(l)) for l in labels)
    return total_dist / total_chars

Full eval on val split

In [9]:
eval_paths, eval_words = [], []
with open("../data/synth/eval_fresh/labels.txt") as f:
    for line in f:
        fname, word = line.strip().split("\t")
        eval_paths.append(f"../data/synth/eval_fresh/{fname}")
        eval_words.append(word)

print(len(eval_paths))

preds = []
for p in eval_paths:
    img = transform(Image.open(p).convert('RGB'))
    preds.append(generate(model, img))

acc = word_accuracy(preds, eval_words)
error_rate = cer(preds, eval_words)
print(f"word accuracy: {acc:.4f}")
print(f"CER: {error_rate:.4f}")

1000
word accuracy: 0.9310
CER: 0.0275


In [10]:
mistakes = [(l, p) for l, p in zip(eval_words, preds) if normalize(l) != normalize(p)]
print(len(mistakes), "mistakes out of", len(eval_words))
for l, p in mistakes[:15]:
    print(f"true: {l:15s}  pred: {p}")

69 mistakes out of 1000
true: nomw5            pred: nomomw5
true: ykmkwpg          pred: ykwpg
true: l6bfldlxo        pred: l6bflxo
true: transformer      pred: transformermer
true: ea5cg4ejb        pred: ea5cg4ea5b
true: transformer      pred: transformermer
true: 5uksb5gx         pred: 5gb5uksb5
true: 051rvh06x        pred: 061rvh06x
true: transformer      pred: transformermer
true: t7fhh2w          pred: t7fhhh2w
true: w6ysms3f1k       pred: w6ysms3flk
true: uxxkg1ve         pred: uxkg1ve
true: 8mkmm2n3         pred: 8m2nkm2n3
true: axkj4f           pred: axkkj4f
true: yl2h63ve7x       pred: yh63ve7x


Dictionary-words VS Random words test

In [11]:

dictionary_words = ["stop", "hello", "world", "notification", "at", "the", "and",
                     "recognition", "transformer", "python", "image", "attention",
                     "network", "training", "model", "vision", "encoder", "decoder"]
dict_pairs = [(l, p) for l, p in zip(eval_words, preds) if l in dictionary_words]
rand_pairs = [(l, p) for l, p in zip(eval_words, preds) if l not in dictionary_words]

print("dict word accuracy:", word_accuracy([p for l,p in dict_pairs], [l for l,p in dict_pairs]))
print("random string accuracy:", word_accuracy([p for l,p in rand_pairs], [l for l,p in rand_pairs]))

dict word accuracy: 0.9674220963172805
random string accuracy: 0.8435374149659864
